# Can we predict `late_delivery`?
Goal: predict at dispatch time whether a consignment will be late, for future consignments.

Design decisions (from checklist.md and columns.csv):
- Each `consignment` appears once, but the goal is forecasting later rows, so the **test set is the latest 20% of rows by `dispatched_on`**.
- CV inside the training rows is an **expanding-window time-series split** (earlier rows train, later rows validate).
- `claim_paid_eur` is compensation paid after the delivery outcome is known, so it is not available at prediction time. It is dropped (it nearly encodes the label). `consignment` is an identifier and is dropped.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, average_precision_score
from sklearn.base import clone
df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.shape, df.dispatched_on.min(), df.dispatched_on.max())
print('late rate by claim>0 (why claim_paid_eur is excluded):')
print(df.groupby(df.claim_paid_eur > 0).late_delivery.mean())

## 1. Hold out the test set (latest 20% of rows) before any modelling

In [ ]:
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(len(train), len(test), train.dispatched_on.max(), test.dispatched_on.min())
print('train late rate', round(train.late_delivery.mean(), 3))

## 2. Exploration (training rows only)
The late rate drifts strongly over time, so time-ordered evaluation matters.

In [ ]:
print(train.groupby(train.dispatched_on.dt.to_period('M')).late_delivery.mean().round(3).to_string())
print(train.groupby('carrier').late_delivery.mean().round(3))

## 3. Candidates, declared before any scoring
Features known at dispatch: `carrier`, `distance_km`, `weight_kg`, `parcels` (set A); set B adds `hour` and `dayofweek` derived from `dispatched_on`. Calendar month / trend is not used as a feature, since the model must extrapolate to later dates.

Candidates:
1. `baseline`: predicts the training prior (no features).
2. `logreg_A_*`, `logreg_B_*`: logistic regression, C in {0.1, 1, 10}.
3. `hgb_A_*`, `hgb_B_*`: HistGradientBoosting, max_depth in {2, 4}, learning_rate 0.05, 200 iterations.

Selection metric: mean ROC AUC over time-series CV folds. Decision threshold: chosen on the pooled out-of-fold CV predictions of the winner to maximise F1. Scaling and one-hot encoding live inside the pipelines, so they are fit within each fold.

In [ ]:
for d in (train, test):
    d['hour'] = d.dispatched_on.dt.hour
    d['dayofweek'] = d.dispatched_on.dt.dayofweek
FA = ['carrier', 'distance_km', 'weight_kg', 'parcels']
FB = FA + ['hour', 'dayofweek']

def pre(fs):
    num = [c for c in fs if c != 'carrier']
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), ['carrier']), ('n', StandardScaler(), num)])

cands = {'baseline': (FA, DummyClassifier(strategy='prior'))}
for nm, fs in [('A', FA), ('B', FB)]:
    for c_ in (0.1, 1, 10):
        cands[f'logreg_{nm}_C{c_}'] = (fs, make_pipeline(pre(fs), LogisticRegression(C=c_, max_iter=1000)))
    for dep in (2, 4):
        cands[f'hgb_{nm}_d{dep}'] = (fs, make_pipeline(pre(fs), HistGradientBoostingClassifier(
            max_depth=dep, learning_rate=0.05, max_iter=200, random_state=0)))
print(list(cands))

## 4. Time-series cross-validation on training rows only

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
y = train.late_delivery.values
res, oof = {}, {}
for nm, (fs, m) in cands.items():
    aucs, pred = [], np.full(len(train), np.nan)
    for tr, va in tscv.split(train):
        mm = clone(m).fit(train.iloc[tr][fs], y[tr])
        p = mm.predict_proba(train.iloc[va][fs])[:, 1]
        pred[va] = p
        aucs.append(roc_auc_score(y[va], p))
    res[nm] = (np.mean(aucs), np.std(aucs))
    oof[nm] = pred
cv = pd.DataFrame(res, index=['auc_mean', 'auc_std']).T.sort_values('auc_mean', ascending=False)
cv.round(4)

## 5. Choose model and threshold on CV only

In [ ]:
best = cv.index[0]
fs, m = cands[best]
mask = ~np.isnan(oof[best])
ths = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y[mask], oof[best][mask] >= t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| CV AUC %.3f' % cv.loc[best, 'auc_mean'], '| threshold', round(thr, 2))
print('baseline CV AUC %.3f' % cv.loc['baseline', 'auc_mean'])

**Recorded choice (fixed before touching the test set):** the model, feature set, hyperparameters and threshold printed above. The test set is scored once below, with no further changes.

## 6. Refit on all training rows; score the test set once

In [ ]:
final = clone(m).fit(train[fs], y)
pt = final.predict_proba(test[fs])[:, 1]
yt = test.late_delivery.values
print('test late rate %.3f' % yt.mean())
print('TEST ROC AUC       : %.3f (baseline 0.500)' % roc_auc_score(yt, pt))
print('TEST avg precision : %.3f (chance = late rate)' % average_precision_score(yt, pt))
yh = pt >= thr
print('TEST @thr  precision %.3f  recall %.3f  F1 %.3f' % (
    precision_score(yt, yh, zero_division=0), recall_score(yt, yh), f1_score(yt, yh)))

## Conclusion
The test output above is the headline. Caveat: the late rate falls sharply over the data period (roughly 30% early on to about 1-2% at the end), so the test period has a very different base rate from training. Ranking ability (AUC) may transfer even if the F1-optimal threshold chosen on earlier data does not, so precision at that threshold should be read with this shift in mind. Only features available at dispatch were used; `claim_paid_eur` was excluded as post-outcome leakage.